# DiTFlow motion probes: Wan versus CogVideoX

Run in the existing Colab environment after updating this repository. This notebook observes the current method; it does not change the motion objective or injection/guidance layers. Full instructions and interpretation: `MOTION_PROBES.md`.

Run one generation cell at a time. The three primary runs use the same Lucia prompt and seed, with optional AMF filters off. Backbone geometry and noise schedules still differ. Observing several blocks adds runtime and VRAM overhead.

In [ ]:
# Use the checkout already set up by notebook.ipynb.
from pathlib import Path
import os
if Path('/content/ditflow').is_dir():
    os.chdir('/content/ditflow')
assert Path('motion_guidance_wan.py').is_file(), 'Change to the ditflow repository first'
!python -m pip install -q matplotlib
!python verify_motion_probe.py
!python verify_wan_port.py

## 1. Wan Euler
Observe blocks 0, 10, 15 and 20. Guidance remains at block 15; KV injection remains at block 0. Default capture steps include the end of guidance and several later steps.

In [ ]:
!python motion_guidance_wan.py -v assets/lucia.mp4 -p "Cat walks in a city lane" --scheduler flowmatch --seed 1 --probe --probe_blocks 0 10 15 20 --output_path probe_runs/lucia_wan_euler

## 2. Wan UniPC
Same settings as above except the scheduler, to check the earlier solver hypothesis.

In [ ]:
!python motion_guidance_wan.py -v assets/lucia.mp4 -p "Cat walks in a city lane" --scheduler unipc --seed 1 --probe --probe_blocks 0 10 15 20 --output_path probe_runs/lucia_wan_unipc

## 3. CogVideoX-5B
Uses the original defaults, including 24 frames at 720×480 and guidance block 20. Wan uses 21 frames at 832×480. Both have six latent frames, but they are not exactly aligned in temporal support. Use an appropriately sized GPU for this model.

In [ ]:
!python motion_guidance.py -v assets/lucia.mp4 -p "Cat walks in a city lane" --seed 1 --probe --probe_blocks 0 15 20 25 --output_path probe_runs/lucia_cog

## 4. Build the comparison
This can run after just one generation. It automatically includes completed or partial traces from the directories listed below. The report uses the latest trace in each directory.

In [ ]:
from probe_report import make_report
from IPython.display import display, HTML
runs = [str(p) for p in [Path('probe_runs/lucia_wan_euler'), Path('probe_runs/lucia_wan_unipc'), Path('probe_runs/lucia_cog')] if list((p / 'probes').glob('*/events.jsonl'))]
assert runs, 'Run at least one probe generation first'
report = make_report(runs, 'probe_comparison')
display(HTML(report.read_text(encoding='utf-8')))

## 5. Inspect a particular intermediate field
Compare the first pre-update guidance field with the conditional denoising field after all updates, at the same step. Arrows use a fixed normalized scale; gray indicates attention confidence. These plots are not optical flow measured from decoded frames.

In [ ]:
from probe_report import plot_capture
import matplotlib.pyplot as plt
run = 'probe_runs/lucia_wan_euler'
plot_capture(run, block=15, stage='guidance', step=9, iteration=0, kind='soft')
plt.show()
plot_capture(run, block=15, stage='denoise_cond', step=9, kind='soft')
plt.show()
plot_capture(run, block=15, stage='final_latent', kind='hard')
plt.show()

## Optional: reproduce the earlier filtered Euler case
This run retains `--flow_max_disp 10` from your earlier example. Keep it separate from the unfiltered cross-backbone comparison.

In [ ]:
# Uncomment to run.
# !python motion_guidance_wan.py -v assets/lucia.mp4 -p "Cat walks in a city lane" --scheduler flowmatch --flow_max_disp 10 --seed 1 --probe --probe_blocks 0 10 15 20 --output_path probe_runs/lucia_wan_euler_filtered

In [ ]:
# Download the report and traces from Colab; excludes large saved embeddings.
import zipfile
archive = Path('probe_comparison/probe_evidence.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in Path('probe_comparison').rglob('*'):
        if p.is_file() and p.suffix != '.zip':
            z.write(p, str(p))
    for run in runs:
        for p in (Path(run) / 'probes').rglob('*'):
            if p.is_file():
                z.write(p, str(p))
# Default: persist in Google Drive and offer a clickable browser download.
# Set False for browser download only (no Drive mount required).
SAVE_PROBE_ZIP_TO_DRIVE = True
import importlib
import colab_utils
importlib.reload(colab_utils)  # Pick up the helper after pulling an updated checkout.
archive_locations = colab_utils.export_probe_archive(
    archive, save_to_drive=SAVE_PROBE_ZIP_TO_DRIVE, drive_subdir='ditflow_probes')
